# TissueMC: GPU rung R1 on a Colab T4

Rung R1 accumulates absorbed weight per layer inside each thread and publishes it once per photon, instead of two global atomic adds per absorption event.

This notebook validates R1, compares it with the CPU on identical photons, benchmarks **R0, then R1, then R0 again in the same session** (so the GPU's own drift is visible), and reads the hardware counters for R0 and R1 with Nsight Compute. It ends with **one summary block** for you to paste back.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU.*
2. Have your **R1 prediction recorded in `LOG.md`** (tell the assistant) before you run the benchmark cells.

Then *Runtime → Run all*. It takes roughly 15 to 25 minutes. Nothing here needs passwords or keys.

In [1]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,clocks.max.sm --format=csv,noheader
!nvcc --version | tail -1

Tesla T4, 580.82.07, 15360 MiB, 1590 MHz
Build cuda_13.0.r13.0/compiler.36424714_0


In [2]:
!git clone -q https://github.com/Raghava-Dwivedi/tissuemc.git
%cd tissuemc
!git log -1 --format='commit %h  %s'

/content/tissuemc
commit a6098ca  feat(gpu): rung R1 (per-photon accumulation of absorbed weight), profiling tool, R0 results and log


In [3]:
!pip install -q cmake ninja numpy matplotlib
!pip install -q -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 17.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for tissuemc-analysis (pyproject.toml) ... done


In [4]:
!cmake -S . -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DTISSUEMC_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 > /dev/null
!cmake --build build 2>&1 | tail -3
!./build/tissuemc --version

[15/17] Linking CXX executable tissuemc
[16/17] Building CXX object CMakeFiles/tissuemc_tests.dir/tests/cpp/test_gpu_emulation.cpp.o
[17/17] Linking CXX executable tissuemc_tests
tissuemc 0.1.0


In [5]:
# 1. Validate rung 1 against the published values and the independent oracle.
!tissuemc-validate --binary build/tissuemc --engine cuda --rung 1 --report results/gpu-r1-validation.json 2>&1 | grep -v '^  note' | tail -45

engine: cuda, transport precision: float64

V1: slab with matched boundary (MCML manual Table 6.1)
  [PASS] rd                ours 0.097406 +/- 0.000112   ref 0.097390   z = +0.14   van de Hulst 1980 (Table 6.1)
  [PASS] rd                ours 0.097406 +/- 0.000112   ref 0.097340   z = +0.18   MCML (Table 6.1)
  [PASS] rd                ours 0.097406 +/- 0.000112   ref 0.097110   z = +0.85   Prahl et al. 1989 (Table 6.1)
  [PASS] tt                ours 0.660980 +/- 0.000175   ref 0.660960   z = +0.11   van de Hulst 1980 (Table 6.1)
  [PASS] tt                ours 0.660980 +/- 0.000175   ref 0.660960   z = +0.08   MCML (Table 6.1)
  [PASS] tt                ours 0.660980 +/- 0.000175   ref 0.661590   z = -1.17   Prahl et al. 1989 (Table 6.1)
  [PASS] conservation      ours -0.000000 +/- 0.000000   ref 0.000000   z = -2969.79   rd + tt + absorbed + rsp = 1
  [PASS] anomalies         ours 0.000000 +/- 0.000000   ref 0.000000   z = +0.00   non-finite weights or non-unit directions
V2: semi

In [6]:
# 2. Same photons through the CPU (float32) and rung 1: they should agree far more closely than noise.
!tissuemc-cross-engine --binary build/tissuemc --b-engine cuda --b-rung 1 --photons 2000000 --report results/gpu-r1-cross-engine.json

v3_three_layer: 2000000 photons, 50 batches, seed 1
  A = cpu float32    B = cuda float32
  device: Tesla T4 (sm_75)
  [PASS] rd: total gap / standard error     1.299e-03  (limit 2.5e-01)
  [PASS] rd: worst per-batch relative gap   7.970e-05  (limit 5.0e-03)
  [PASS] tt: total gap / standard error     8.819e-03  (limit 2.5e-01)
  [PASS] tt: worst per-batch relative gap   2.425e-04  (limit 5.0e-03)
  [PASS] events: relative gap               2.328e-06  (limit 1.0e-03)
  [PASS] anomalies (both runs)              0.000e+00  (limit 0.0e+00)

engines agree on identical photons


In [7]:
# 3. Benchmarks, back to back in this session: R0, R1, then R0 again to show session drift.
!tissuemc-bench --engine cuda --rung 0 --binary build/tissuemc --output results/gpu-r0-session.json 2>&1 | tail -4
!tissuemc-bench --engine cuda --rung 1 --binary build/tissuemc --output results/gpu-r1.json 2>&1 | tail -4
!tissuemc-bench --engine cuda --rung 0 --binary build/tissuemc --output results/gpu-r0-again.json 2>&1 | tail -4

  run 6/6: kernel 11.268 s

wrote results/gpu-r0-session.json
Tesla T4 (sm_75): 549.0 million events/s (median kernel)
  run 6/6: kernel 10.359 s

wrote results/gpu-r1.json
Tesla T4 (sm_75): 603.1 million events/s (median kernel)
  run 6/6: kernel 11.382 s

wrote results/gpu-r0-again.json
Tesla T4 (sm_75): 545.9 million events/s (median kernel)


In [8]:
# 4. Hardware counters (Nsight Compute) for both rungs: what actually limits each kernel.
!tissuemc-profile --rung 0 --binary build/tissuemc --output results/gpu-r0-profile.json
!tissuemc-profile --rung 1 --binary build/tissuemc --output results/gpu-r1-profile.json

rung 0, 200000 photons (tissue_g09.mci); key counters:
  Duration                                   1,046,848,480 ns   kernel time
  Compute (SM) Throughput                           29.55 %   how busy the arithmetic units are (% of peak)
  Memory Throughput                                 10.98 %   how busy the memory system is (% of peak)
  L2 Cache Throughput                                9.34 %   L2 busy, where global atomics are resolved (% of peak)
  Avg. Active Threads Per Warp                       4.75   threads active per warp instruction, of 32 (divergence)
  Warp Cycles Per Issued Instruction                22.83 cycle   average stall between issued instructions
  Achieved Occupancy                                85.67 %   warps resident per SM, % of the maximum
  Theoretical Occupancy                               100 %   the limit set by registers and block size
rung 1, 200000 photons (tissue_g09.mci); key counters:
  Duration                                    929,217,5

In [ ]:
import sys

sys.path.insert(0, "python")
import json
import pathlib
import subprocess

from tissuemc_analysis.ncu_profile import KEY_METRICS


def load(name):
    p = pathlib.Path("results") / name
    return json.loads(p.read_text()) if p.exists() else None


def bench_summary(b):
    if not b:
        return None
    return {
        "median_kernel_s": round(b["kernel"]["median"], 3),
        "kernel_seconds": [round(x, 3) for x in b["kernel_seconds"]],
        "million_events_per_s": round(b["events_per_second"] / 1e6, 1),
        "validated_first": b["validated_before_measuring"],
    }


def profile_summary(p):
    if not p:
        return None
    out = {}
    for section, metric, _ in KEY_METRICS:
        entry = p["sections"].get(section, {}).get(metric)
        if entry:
            out[metric] = f"{entry['value']} {entry['unit']}".strip()
    return out


val, cross = load("gpu-r1-validation.json"), load("gpu-r1-cross-engine.json")
commit = subprocess.run(
    ["git", "log", "-1", "--format=%H"], capture_output=True, text=True
).stdout.strip()
r1 = load("gpu-r1.json")
summary = {
    "commit": commit,
    "device": r1["device"] if r1 else None,
    "validation_r1": None
    if not val
    else {
        "checks": len(val),
        "passed": sum(r["passed"] for r in val),
        "failed": [
            f"{r['case']} {r['quantity']} z={r['z']:+.2f} ({r['source']})"
            for r in val
            if not r["passed"]
        ],
    },
    "cross_engine_r1": None
    if not cross
    else {
        "agree": cross["agree"],
        "gaps": {g["name"]: [g["value"], g["limit"]] for g in cross["gaps"]},
    },
    "benchmark": {
        "r0_first": bench_summary(load("gpu-r0-session.json")),
        "r1": bench_summary(r1),
        "r0_again": bench_summary(load("gpu-r0-again.json")),
    },
    "counters": {
        "r0": profile_summary(load("gpu-r0-profile.json")),
        "r1": profile_summary(load("gpu-r1-profile.json")),
    },
}
print("===== R1 SUMMARY =====")
print(json.dumps(summary, indent=1))